#### 提示词模板：变量插入到模板中
模板类型：

    1.PromptTemplate:生成字符串提示 

    2.FewShotPromptTemplate:样本提示词模板

    3.ChatPromptTemplate:聊天提示，组合各种角色的消息模板

    4.HumanMessagePromptTemplate:4-7 各类消息模板

    5.AIMessagePromptTemplate

    6.SystemMessagePromptTemplate

    7.ChatMessagePromptTemplate

    8.PipelinePrompt:管道提示词模板

    9.自定义模板

In [ ]:
from langchain_core.prompts import PromptTemplate

text = """
{name}是一名{job},你{age}岁
"""

# PromptTemplate的使用

# 1）获取实例
# # 必须指明 input_variables template
# # # prompt_template = PromptTemplate(input_variables=["name"], template="你的名字是{name}")
prompt_template = PromptTemplate.from_template(template="你的名字是{name},你的工作是{job},你{age}岁",partial_variables={"age":"18"})
prompt_2 = PromptTemplate.from_template(text,partial_variables={"age":"18"})
# 2）两种特殊结构的使用
templates = PromptTemplate.from_template(
    template="你的名字是{name},你的工作是{job},你{age}岁"
).partial(name="小王")


# 3）给变量赋值的两种方式：1）使用format方法；2）使用invoke方法
# # 填充模板的变量
prompt = prompt_template.format(name="小王", job="程序员")
prompt2 = prompt_2.format(name="小王", job="程序员")
prompt3 = templates.format(job="程序员",age="20")
print(prompt)
print(prompt2)
print(prompt3)

# # # invoke()
print(type(prompt))
# 1. 创建模板并预填充 name="小李"
templates2 = PromptTemplate.from_template(
    template="你的名字是{name},你的工作是{job},你{age}岁"
).partial(name="小李")

# 2. 使用 invoke 方法填充剩余变量
# 注意：必须使用字典格式传参
prompt4 = templates2.invoke({"job": "程序员", "age": "19"})

print(prompt4)
# 输出: 你的名字是小李,你的工作是程序员,你19岁

print(type(prompt4))
# 输出: <class 'langchain_core.prompts.prompt.StringPromptValue'>

# 4）结合大模型的使用
import os
import dotenv
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
api_key = os.getenv("openai_api_key")
base_url = os.getenv("openai_base_url")

chat_model = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=api_key,
    base_url=base_url,
    temperature=0.8,
    max_tokens=1000
)
response = chat_model.invoke(prompt4)
print(response.content)
print(type(response))

In [ ]:
# FewShotPromptTemplate: 与PromptTemplate一起使用
# 提供示例
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
api_key = os.getenv("openai_api_key")
base_url = os.getenv("openai_base_url")
chat_model_stream = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=api_key,
    base_url=base_url,
    temperature=0.8,
    max_tokens=1000,
    streaming=True
)
# 提供示例

from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.prompts  import PromptTemplate
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)
example= [
    {
        "input": "What is the capital of France?",
        "output": "Paris"
    },
    {
        "input": "What is the capital of Germany?",
        "output": "Berlin"
    }
]
few_shot = FewShotPromptTemplate(
    examples=example,
    example_prompt=example_prompt,
    suffix="Input: {input}\nOutput: ",# 后缀
    input_variables=["input"],
)
few_shot_prompt = few_shot.format(input="What is the capital of Japan?")
response = chat_model_stream.invoke(few_shot_prompt)
print(response)

In [ ]:
# 提示词模板之ChatPromptTemplate的使用
# # 1、实例化的方式（两种方式：使用构造方法、from_messages
from langchain_core.prompts import ChatPromptTemplate

chat_templates = ChatPromptTemplate(
    messages=[
    ("system", "You are a helpful assistant that translates {input_language} to {output_language}"),
    ("human", "{text}"),
],
input_variables=["input_language", "output_language", "text"]
)
input_language = "中文"
output_language = "英文"
text = "你好"

# 2、调用提示词模板的几种方法：invoke() format() format_messages() format_prompt()
# #
# invoke()：传入的是字典，返回ChatPromptValue
chat_prompt = chat_templates.invoke(input={"input_language": input_language, "output_language": output_language, "text": text})
print(chat_prompt)
print(type(chat_prompt))

chat_prompt_messages = chat_prompt.to_messages()
print(chat_prompt_messages)
print(type(chat_prompt_messages))

chat_prompt_str = chat_prompt.to_string()
print(chat_prompt_str)
print(type(chat_prompt_str))
# format(:传入变量的值，返回str
# format_messages(：传入变量的值，返回消息构成的list
# format_prompt()：传入变量的值，返回ChatPromptvalue

# 3、更丰富的实例化参数类型
# 4、结合LLM
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
dotenv.load_dotenv()
api_key = os.getenv("openai_api_key")
base_url = os.getenv("openai_base_url")
chat_model_stream = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=api_key,
    base_url=base_url,
    temperature=0.8,
    max_tokens=1000,
    streaming=True
)
response = chat_model_stream.invoke(chat_prompt)
print(response.content)
# 5、插入消息列表：MessagePlaceholder
# # 不确定消息提示词模板使用什么角色，或者洗完再格式化过程中插入消息列表时
from langchain_core.prompts.chat import MessagesPlaceholder
chat_prompt_templates = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates {input_language} to {output_language}"),
    MessagesPlaceholder(variable_name="history"),# history
])
chat_prompt = chat_prompt_templates.format_prompt(input_language="中文", output_language="英文",history=[HumanMessage(content="年")])
response = chat_model_stream.invoke(chat_prompt)
print(response.content)

In [ ]:
# PipelinePromptTemplate 按顺序组合成处理管道
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.pipeline import PipelinePromptTemplate # 0.3.22 版本后已弃用